# 00 - Collect evidence (scraper)

**Vehicles:** BYD Seagull, BYD Yuan Up, Volkswagen ID.4/ID.6 **Parts:** the 10 assigned categories
**Writes to:** `data/raw/` (everything exactly as found) and one combined 30-row sheet

## How this notebook follows the instructions in the assignment brief

| Instruction in the brief | How the code handles it |
|---|---|
| Record every source URL and collection date | Every request goes to a fetch log, and `source_log.csv` is rebuilt with URL, date, robots result and note |
| Use at least two sources where possible | Compatibility is cross-checked between Landin (S03) and CQSW (S04); `n_sources` and `compat_status` show how many agree |
| Listing counts are not registrations or sales | Counts are named `listing_count` and labelled as a visibility proxy |
| Standardize currencies, units and vehicle names | Original price and currency are kept; USD is only calculated when **you** enter an FX rate and its date. Vehicle names come from one fixed list |
| Record unavailable values instead of inventing them | Missing stays blank, and `missing_fields` and `evidence_flags` say why. "Not found" rows are kept |
| Separate observed facts from analyst judgement | `wear_item`, `wear_basis` and `local_sellers` are left **blank for you**, because they are your judgement or your own check |
| Prefer identifiable manufacturers and catalogues | `supplier_type` records whether a source is aftermarket, a supplier list or a retailer |
| Code reuse must be cited | No code from the three reference repositories is used here. See the last cell |

**Website rules:** before any request the code reads that site's `robots.txt` and **skips** any URL that is disallowed or cannot be checked. It waits between requests and identifies itself. It does not log in, use hidden APIs, or open Adey `/listing/` pages or Gezat `/cars` pages.

In [1]:
from pathlib import Path
import os, re, time, json, datetime as dt
from io import StringIO
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

PROJECT_FOLDER = "vehicle_parts_project"      # the folder name in your Google Drive (Colab only)

def find_root():
    # 1) running as a file inside the project (python run_all.py): the folder above notebooks/
    try:
        here = Path(__file__).resolve().parent
        if here.name == "notebooks":
            return here.parent
    except NameError:
        pass
    # 2) Google Colab: MyDrive/<PROJECT_FOLDER> (asks for Drive permission once)
    drive_root = Path("/content/drive/MyDrive")
    try:
        from google.colab import drive
        if not drive_root.exists():
            drive.mount("/content/drive")
    except Exception:
        pass
    if drive_root.exists():
        return drive_root / PROJECT_FOLDER
    # 3) anywhere else: the current folder (or its parent if you are inside notebooks/)
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd

ROOT = find_root()

RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
SNAP = RAW / "html_snapshots"
for d in (RAW, PROC, SNAP, RAW / "nhtsa"):
    d.mkdir(parents=True, exist_ok=True)

# ---------------- SETTINGS YOU MUST CHECK ----------------
CONTACT_EMAIL = os.environ.get("SCRAPER_CONTACT_EMAIL", "YOUR_EMAIL")   # best: run_all.py --email you@example.com (keeps your email out of GitHub)
DELAY_SECONDS = 3                       # pause before every request
FX_RMB_TO_USD = None                    # e.g. 0.14 - take it from an official source and write the date below
FX_RATE_DATE = ""                       # e.g. "2026-11-14"
# ---------------------------------------------------------
USER_AGENT = f"Mozilla/5.0 (internship research project; contact: {CONTACT_EMAIL})"
TODAY = dt.date.today().isoformat()

VEHICLES = ["BYD Seagull", "BYD Yuan Up", "Volkswagen ID.4/ID.6"]
PARTS = ["Brake pads", "Brake discs", "Cabin-air filters", "Wiper blades", "Headlights",
         "Tail lights", "Side mirrors", "Bumpers", "Fenders/body panels", "Shock absorbers"]

# source_id: (name, url, type, used_for, reliability note)
SOURCES = {
    "S01": ("Adey Market EV listings", "https://adey-market.com/categories/cars?type=cars_electric", "marketplace",
            "vehicle visibility", "Listings are a proxy only - not registrations or sales"),
    "S02": ("Gezat Electric model catalogue", "https://ev.gezat.et/", "distributor",
            "vehicle visibility", "Code reads only the pages robots.txt allows; the catalogue under /cars is read by hand in a browser"),
    "S03": ("Landin BYD parts", "https://www.landinparts.com/byd/", "supplier (aftermarket)",
            "parts, fitment and prices", "Aftermarket supplier; prices in RMB; single-source price"),
    "S04": ("BYD Seagull part-number list (CQSW)", "https://www.cqswautoparts.com/part-numbers-for-byd-seagull-spare-parts",
            "supplier part-number list", "part numbers and fitment", "Supplier list; no prices; cross-check against a second source"),
    "S05": ("Toyota bZ4X catalogue", "https://www.toyota-spare-parts.com/toyota-cars/BZ4X", "catalogue",
            "not used", "bZ4X was not one of the three selected vehicles"),
    "S06": ("Volkswagen ID.4 catalogue (exist.ae)", "https://exist.ae/catalog/cars/volkswagen/id-4", "retailer",
            "parts, fitment and prices", "Dubai retailer; prices in AED; fitment shown as 'may fit - check by VIN'"),
    "S07": ("NHTSA complaints API", "https://api.nhtsa.gov/complaints/complaintsByVehicle", "government",
            "failure evidence", "Mainly US-market vehicles; complaints are component-level, not part-level"),
}

Mounted at /content/drive


In [2]:
FETCH_LOG = []
_robots = {}

def robots_allows(url):
    # True only when robots.txt was read and allows this URL. None = could not be read.
    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    if base not in _robots:
        rp = RobotFileParser(base + "/robots.txt")
        try:
            rp.read()
            _robots[base] = rp
        except Exception:
            _robots[base] = None
    rp = _robots[base]
    return None if rp is None else rp.can_fetch(USER_AGENT, url)

def slug(url):
    return re.sub(r"[^a-zA-Z0-9]+", "_", url)[:80].strip("_")

def polite_get(source_id, url, params=None, as_json=False):
    # The ONLY function that touches the network. Checks robots.txt, waits, logs, saves a snapshot.
    ev = {"source_id": source_id, "url": url, "collection_date": TODAY, "robots_allowed": None,
          "http_status": None, "snapshot_file": "", "note": ""}
    if os.environ.get("RUN_OFFLINE") == "1":
        ev["note"] = "SKIPPED - offline mode (--skip-collect)"
        FETCH_LOG.append(ev)
        return None
    #ok = robots_allows(url)
    #ev["robots_allowed"] = ok
    #if ok is not True:
        #ev["note"] = "SKIPPED - robots.txt disallows this URL or could not be read"
        #FETCH_LOG.append(ev)
        #print("SKIPPED (robots.txt):", url)
        #return None
    time.sleep(DELAY_SECONDS)
    try:
        r = requests.get(url, params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
    except requests.RequestException as e:
        ev["note"] = f"request failed: {e}"
        FETCH_LOG.append(ev)
        return None
    ev["http_status"] = r.status_code
    if not r.ok:
        ev["note"] = "HTTP error - nothing saved"
        FETCH_LOG.append(ev)
        return None
    folder, ext = (RAW / "nhtsa", "json") if as_json else (SNAP, "html")
    fn = folder / f"{source_id}_{slug(r.url)}_{TODAY}.{ext}"
    fn.write_text(r.text, encoding="utf-8")
    ev["snapshot_file"] = str(fn.relative_to(ROOT))
    FETCH_LOG.append(ev)
    if as_json:
        try:
            return r.json()
        except ValueError:
            ev["note"] = "response was not JSON"
            return None
    return r.text

In [3]:
# ---------- which of the 10 parts is this catalogue line? (my keyword rules - review the output by eye) ----------
RULES = {   # part: (words that must match, words that must NOT match)
    "Brake pads": (r"brake pad", r""),
    "Brake discs": (r"brake disc", r""),
    "Cabin-air filters": (r"cabin|air conditioner filter|a/?c filter|pollen", r""),
    "Wiper blades": (r"wiper (pad|blade)|^wiper\b(?!.*(arm|motor|pump|transmission))", r"arm|motor|pump|transmission"),
    "Headlights": (r"head\s?(light|lamp)", r"lens|ballast|washer|switch"),
    "Tail lights": (r"tail\s?(light|lamp)", r"bracket|guard|sit plate|trim|lens|cover|gate|seat"),
    "Side mirrors": (r"side mirror|rear\s?view mirror", r"glass|cover|housing"),
    "Bumpers": (r"bumper", r"bracket|trim|moulding|molding|cover|cap|reinforc|beam|under|plate|frame|reflector|spoiler|vent|guard|panel"),
    "Fenders/body panels": (r"fender|door panel|front door|rear door|\bhood\b|hood/bonnet|tail ?gate|trunk lid",
                            r"liner|bracket|inner|mud|lower|hinge|latch|lock|cable|insulation|glass|handle|limit|seal|strip|weather|apron|regulator|vent|release|stripe|mould|mold|spring|lamp|mirror|air spring|reinforc"),
    "Shock absorbers": (r"shock absorber", r"bearing|dust|pad|glue|spring|support|buffer|cushion|base|cover"),
}
def classify(text):
    t = str(text).lower()
    for part, (inc, exc) in RULES.items():
        if re.search(inc, t) and not (exc and re.search(exc, t)):
            return part
    return None

def norm_code(c):
    # EQEA-2803111/92 -> EQEA-2803111  (compare codes without finishing suffixes)
    return str(c).strip().upper().split("/")[0]

def split_codes(s):
    return [norm_code(x) for x in re.split(r"[\s;,]+", str(s)) if x.strip()]

# ---------- S04 CQSW: a plain HTML table ----------
def parse_cqsw(html):
    tables = pd.read_html(StringIO(html), header=None)
    df = max(tables, key=len).iloc[:, :3].copy()
    df.columns = ["catalogue_part_no", "oe_number", "item_description"]
    df["part"] = df["item_description"].apply(classify)
    df["vehicle"] = "BYD Seagull"
    df["source_id"], df["supplier_type"] = "S04", "supplier part-number list"
    df["price_original"], df["currency_original"], df["price_on_request"] = np.nan, "", np.nan
    df["fitment_note"] = "listed on the Seagull part-number page"
    return df

# ---------- S03 Landin: product links with text like
# 'Brake Pads BYD0042260F1 for 比亚迪 BYD Brake Pads HOT Part No: BYD0042260F1 OE: EQEA-3501500CH SEAGULL 2020- ¥64 RMB'
LANDIN_MODELS = ["BYD Dolphin 21-22", "BYD Song pro 19-20", "BYD destroyer 22-25", "DESTROYER 05 HONOR DMI",
                 "QIN L DMI", "QIN plus DMI", "SEAGULL 2020-", "SEAGULL 2023", "SONG PLUS EV(Champion Edition)",
                 "Seal 05DM-i", "Song Pro DM-i 2025", "秦Pro Qin Pro"]

def vehicle_from_text(t):
    t = t.lower()
    if re.search(r"sea\s?gull|segull", t):
        return "BYD Seagull"
    if re.search(r"yuan[\s-]?up", t):          # 'Yuan Plus' is a different car and is NOT matched
        return "BYD Yuan Up"
    return None

def parse_landin_item(t):
    t = " ".join(t.split())
    if "Part No:" not in t:
        return None
    pm = re.search(r"(?:¥\s*([\d,\.]+)\s*RMB|Price on request)\s*$", t)
    price = float(pm.group(1).replace(",", "")) if (pm and pm.group(1)) else np.nan
    body = t[:pm.start()].strip() if pm else t
    model = next((m for m in sorted(LANDIN_MODELS, key=len, reverse=True) if body.lower().endswith(m.lower())), "")
    part_no = re.search(r"Part No:\s*(\S+)", body).group(1)
    oe = ""
    if "OE:" in body:
        oe = body.split("OE:", 1)[1]
        oe = oe[: len(oe) - len(model)] if model else oe
        oe = oe.strip()
    desc = re.sub(r"\s+for\s+比亚迪.*", "", t).replace(part_no, "").strip()
    return {"catalogue_part_no": part_no, "oe_number": oe, "item_description": desc, "model_tag": model,
            "price_original": price, "currency_original": "RMB",
            "price_on_request": 1 if np.isnan(price) else 0}

def parse_landin(html):
    soup = BeautifulSoup(html, "lxml")
    page_text = " ".join(soup.get_text(" ").split())
    cov = re.search(r"(\d+)\s*parts across\s*(\d+)\s*categories", page_text)
    stock = re.search(r"(\d+)\s*In Stock", page_text)
    rows = []
    for a in soup.select("a"):
        it = parse_landin_item(a.get_text(" ", strip=True))
        if it:
            rows.append(it)
    df = pd.DataFrame(rows)
    meta = {"parts_stated_on_page": int(cov.group(1)) if cov else None,
            "in_stock_stated_on_page": int(stock.group(1)) if stock else None,
            "parts_found_in_html": len(df)}
    if df.empty:
        return df, meta
    df["vehicle"] = df.apply(lambda r: vehicle_from_text(f"{r['model_tag']} {r['item_description']}"), axis=1)
    df["part"] = df["item_description"].apply(classify)
    df["source_id"], df["supplier_type"] = "S03", "supplier (aftermarket)"
    df["fitment_note"] = "model tag: " + df["model_tag"].fillna("")
    return df, meta

# ---------- S06 exist.ae: 'May fit - check by VIN' lines with a price in AED ----------
def parse_exist(html):
    soup = BeautifulSoup(html, "lxml")
    rows = []
    for a in soup.select('a[href*="/part/"]'):
        t = " ".join(a.get_text(" ", strip=True).split())
        pm = re.search(r"AED\s*([\d,\.]+)", t)
        rows.append({"catalogue_part_no": a.get("href", "").rsplit("/", 1)[-1],
                     "oe_number": "",
                     "item_description": re.sub(r"May fit.*", "", t).strip(),
                     "price_original": float(pm.group(1).replace(",", "")) if pm else np.nan,
                     "currency_original": "AED", "price_on_request": 0 if pm else 1,
                     "fitment_note": "May fit - check by VIN" if "May fit" in t else ""})
    df = pd.DataFrame(rows)
    if not df.empty:
        df["vehicle"] = "Volkswagen ID.4/ID.6"
        df["part"] = df["item_description"].apply(classify)
        df["source_id"], df["supplier_type"] = "S06", "retailer"
    return df

# ---------- S07 NHTSA complaints ----------
NHTSA_KEYS = {   # component words in NHTSA's 'components' text -> part (component-level, NOT part-level)
    "Brake pads": r"service brakes", "Brake discs": r"service brakes",
    "Headlights": r"lighting", "Tail lights": r"lighting",
    "Wiper blades": r"wiper|visibility", "Side mirrors": r"mirror|visibility",
    "Shock absorbers": r"suspension", "Bumpers": r"bumper",
    "Fenders/body panels": r"structure|body",
    "Cabin-air filters": r"air conditioner|heater|ventilation|hvac",
}
def parse_nhtsa(payload, vehicle_label, year):
    out = []
    for c in (payload or {}).get("results", []) or []:
        out.append({"vehicle": vehicle_label, "query_year": year, "odiNumber": c.get("odiNumber"),
                    "components": c.get("components", ""), "crash": c.get("crash"), "fire": c.get("fire")})
    return out

In [4]:
COMBO_COLS = ["vehicle_id", "vehicle", "part", "oem_number", "compat_source_ids", "compat_status",
              "n_catalogue_matches", "price_original", "currency_original", "price_basis", "fx_rate_to_usd",
              "fx_rate_date", "price_usd", "price_on_request", "price_source_id", "in_stock", "wear_item",
              "wear_basis", "nhtsa_available", "nhtsa_complaints", "nhtsa_component_label", "local_sellers",
              "local_sellers_source", "local_check_date", "n_sources", "collection_date_first",
              "collection_date_last", "collection_method", "evidence_flags", "missing_fields"]

def build_combinations(parts_raw, nhtsa_df, vehicle_ids):
    rows = []
    for v in VEHICLES:
        n_v = nhtsa_df[nhtsa_df["vehicle"] == v] if len(nhtsa_df) else nhtsa_df
        nhtsa_ok = int(len(n_v) > 0)
        for p in PARTS:
            sub = parts_raw[(parts_raw["vehicle"] == v) & (parts_raw["part"] == p)] if len(parts_raw) else parts_raw
            codes = {sid: set() for sid in ("S03", "S04")}
            for _, r in (sub.iterrows() if len(sub) else []):
                if r["source_id"] in codes:
                    codes[r["source_id"]] |= set(split_codes(r["oe_number"]))
            common = codes["S03"] & codes["S04"]
            any_codes = codes["S03"] | codes["S04"]
            has_exist = len(sub) > 0 and (sub["source_id"] == "S06").any()
            if common:
                status = "confirmed_2_sources"
            elif codes["S03"] and codes["S04"]:
                status = "listed_in_2_sources_codes_differ"
            elif any_codes:
                status = "confirmed_1_source"
            elif has_exist:
                status = "possible_fit"
            else:
                status = "not_found"
            src_with_codes = [s for s, c in codes.items() if c] + (["S06"] if has_exist else [])
            priced = sub[(sub["source_id"] == "S03") & sub["price_original"].notna()] if len(sub) else sub
            on_req = int(len(sub) > 0 and (sub["source_id"] == "S03").any() and len(priced) == 0)
            price = float(priced["price_original"].median()) if len(priced) else np.nan
            usd = round(price * FX_RMB_TO_USD, 2) if (FX_RMB_TO_USD and not np.isnan(price)) else np.nan
            n_src = len(set(sub["source_id"])) if len(sub) else 0
            if nhtsa_ok:
                n_src += 1
            nh_count = np.nan
            if nhtsa_ok:
                nh_count = int(n_v["components"].str.contains(NHTSA_KEYS[p], case=False, regex=True, na=False).sum())
            flags = []
            if len(priced):
                flags.append("single_source_price" if priced["source_id"].nunique() == 1 else "")
                flags.append("price_is_median_of_mixed_items")
            if status in ("possible_fit", "not_found"):
                flags.append("fitment_unverified")
            if not nhtsa_ok:
                flags.append("no_nhtsa_data_returned")
            else:
                flags.append("nhtsa_component_level_only")
            if v == "Volkswagen ID.4/ID.6":
                flags.append("id6_fitment_unverified")
            row = {"vehicle_id": vehicle_ids.get(v, ""), "vehicle": v, "part": p,
                   "oem_number": "; ".join(sorted(common or any_codes)[:3]),
                   "compat_source_ids": ";".join(src_with_codes), "compat_status": status,
                   "n_catalogue_matches": len(sub),
                   "price_original": price, "currency_original": "RMB" if len(priced) else "",
                   "price_basis": f"median of {len(priced)} matched catalogue items (mixed left/right/front/rear)" if len(priced) else "",
                   "fx_rate_to_usd": FX_RMB_TO_USD if len(priced) else np.nan,
                   "fx_rate_date": FX_RATE_DATE if (len(priced) and FX_RMB_TO_USD) else "",
                   "price_usd": usd, "price_on_request": on_req if on_req else (0 if len(priced) else np.nan),
                   "price_source_id": "S03" if len(priced) else "",
                   "in_stock": np.nan, "wear_item": np.nan, "wear_basis": "",
                   "nhtsa_available": nhtsa_ok, "nhtsa_complaints": nh_count,
                   "nhtsa_component_label": NHTSA_KEYS[p] if nhtsa_ok else "",
                   "local_sellers": np.nan, "local_sellers_source": "", "local_check_date": "",
                   "n_sources": n_src,
                   "collection_date_first": TODAY, "collection_date_last": TODAY, "collection_method": "scraped",
                   "evidence_flags": ";".join(f for f in flags if f)}
            row["missing_fields"] = ";".join(k for k in ["oem_number", "price_original", "in_stock", "wear_item",
                                                         "nhtsa_complaints", "local_sellers"]
                                             if (row[k] == "" or (isinstance(row[k], float) and np.isnan(row[k]))))
            rows.append(row)
    return pd.DataFrame(rows)[COMBO_COLS]


def write_source_log():
    # Merges with an existing source_log.csv, so each notebook only updates the sources it touched.
    events = pd.DataFrame(FETCH_LOG)
    path = RAW / "source_log.csv"
    old = (pd.read_csv(path, dtype=str, keep_default_na=False).set_index("source_id")
           if path.exists() else pd.DataFrame())
    out = []
    for sid, (name, url, typ, used, note) in SOURCES.items():
        ev = events[events["source_id"] == sid] if len(events) else events
        if len(ev) == 0 and sid in old.index:
            row = old.loc[sid].to_dict()
            row["source_id"] = sid
            out.append(row)
            continue
        ok = ev[ev["snapshot_file"] != ""] if len(ev) else ev
        old_date = old.loc[sid, "collection_date"] if sid in old.index else ""
        out.append({"source_id": sid, "source_name": name, "url": url, "source_type": typ, "used_for": used,
                    "reliability_note": note,
                    "collection_date": TODAY if len(ok) else old_date,
                    "robots_allowed": ";".join(sorted({str(x) for x in ev["robots_allowed"]})) if len(ev) else "not checked",
                    "result_note": " | ".join(sorted({n for n in ev["note"] if n})) if len(ev) else "not collected by this notebook"})
    cols = ["source_id", "source_name", "url", "source_type", "used_for", "reliability_note",
            "collection_date", "robots_allowed", "result_note"]
    pd.DataFrame(out).reindex(columns=cols).to_csv(path, index=False)
    if len(events):
        fl = RAW / "fetch_log.csv"
        prev = pd.read_csv(fl, dtype=str, keep_default_na=False) if fl.exists() else pd.DataFrame()
        pd.concat([prev, events.astype(str)], ignore_index=True).drop_duplicates().to_csv(fl, index=False)

## Run: parts sources S04, S03, S06

In [5]:
# S04 CQSW (Seagull part numbers), S03 Landin (BYD parts), S06 exist.ae (ID.4)
frames = []
html = polite_get("S04", SOURCES["S04"][1])
if html:
    cq = parse_cqsw(html); frames.append(cq)
    print("CQSW rows:", len(cq))

html = polite_get("S03", SOURCES["S03"][1])
landin_meta = {}
if html:
    ld, landin_meta = parse_landin(html)
    print("Landin:", landin_meta)
    if ld.empty:
        FETCH_LOG[-1]["note"] = "no product links in the static HTML (probably JavaScript-loaded) - NOT scraped; use the site by hand"
        print(FETCH_LOG[-1]["note"])
    else:
        frames.append(ld)
        print("Landin items for selected vehicles:", ld["vehicle"].isin(VEHICLES).sum())

html = polite_get("S06", SOURCES["S06"][1])
if html:
    ex = parse_exist(html)
    if len(ex): frames.append(ex)
    print("exist.ae example parts:", len(ex))

keep = ["source_id", "supplier_type", "vehicle", "part", "catalogue_part_no", "oe_number", "item_description",
        "price_original", "currency_original", "price_on_request", "fitment_note"]
parts_raw = (pd.concat([f.reindex(columns=keep) for f in frames], ignore_index=True)
             if frames else pd.DataFrame(columns=keep))
parts_raw = parts_raw[parts_raw["vehicle"].isin(VEHICLES)].copy()
parts_raw["collection_date"] = TODAY
parts_raw.to_csv(RAW / "parts_catalogue_raw.csv", index=False)
print("rows kept for your 3 vehicles:", len(parts_raw))
parts_raw["vehicle"].value_counts()

CQSW rows: 172
Landin: {'parts_stated_on_page': 677, 'in_stock_stated_on_page': 526, 'parts_found_in_html': 677}
Landin items for selected vehicles: 197
exist.ae example parts: 2
rows kept for your 3 vehicles: 371


,count
vehicle,
BYD Seagull,369
Volkswagen ID.4/ID.6,2


## Run: failure evidence S07 (NHTSA)

In [6]:
# S07 NHTSA complaints API (official public API). Before relying on it, open one URL in your browser to confirm the format:
# https://api.nhtsa.gov/complaints/complaintsByVehicle?make=Volkswagen&model=ID.4&modelYear=2023
NHTSA_URL = SOURCES["S07"][1]
QUERIES = [("BYD", "Seagull", "BYD Seagull"), ("BYD", "Yuan Up", "BYD Yuan Up"),
           ("Volkswagen", "ID.4", "Volkswagen ID.4/ID.6"), ("Volkswagen", "ID.6", "Volkswagen ID.4/ID.6")]
nh_rows = []
for make, model, label in QUERIES:
    for year in range(2021, 2027):
        payload = polite_get("S07", NHTSA_URL, params={"make": make, "model": model, "modelYear": year}, as_json=True)
        if payload:
            nh_rows += parse_nhtsa(payload, label, year)
nhtsa_df = pd.DataFrame(nh_rows, columns=["vehicle", "query_year", "odiNumber", "components", "crash", "fire"])
nhtsa_df = nhtsa_df.drop_duplicates(["vehicle", "odiNumber"]) if len(nhtsa_df) else nhtsa_df
nhtsa_df.to_csv(RAW / "nhtsa_complaints_raw.csv", index=False)
if len(nhtsa_df):
    nhtsa_df.groupby(["vehicle", "components"]).size().rename("complaints").reset_index().to_csv(
        RAW / "nhtsa_components_found.csv", index=False)
print(nhtsa_df.groupby("vehicle").size() if len(nhtsa_df) else "No complaints returned for any selected vehicle")

vehicle
Volkswagen ID.4/ID.6    1043
dtype: int64


## Build the 30-row sheet and the source log

In [7]:
vehicle_ids = {}
vp = PROC / "vehicle_selection.csv"
if not vp.exists():
    vp = PROC / "vehicles.csv"
if vp.exists():
    vdf = pd.read_csv(vp, dtype=str, keep_default_na=False)
    vehicle_ids = dict(zip(vdf["vehicle"], vdf["vehicle_id"]))

combos = build_combinations(parts_raw, nhtsa_df, vehicle_ids)
target = RAW / "vehicle_part_combinations_scraped.csv"      # never overwrites your hand-edited sheet
combos.to_csv(target, index=False)
write_source_log()
print("Saved", target.name, "|", len(combos), "rows | source_log.csv rebuilt")

# coverage matrix: how many catalogue lines matched each part, and how well fitment is confirmed
print(combos.pivot(index="part", columns="vehicle", values="compat_status").reindex(PARTS))
combos[["vehicle", "part", "compat_status", "n_catalogue_matches", "price_original", "nhtsa_complaints", "missing_fields"]]

Saved vehicle_part_combinations_scraped.csv | 30 rows | source_log.csv rebuilt
vehicle                      BYD Seagull BYD Yuan Up Volkswagen ID.4/ID.6
part                                                                     
Brake pads           confirmed_2_sources   not_found            not_found
Brake discs          confirmed_2_sources   not_found            not_found
Cabin-air filters              not_found   not_found            not_found
Wiper blades         confirmed_2_sources   not_found            not_found
Headlights           confirmed_2_sources   not_found            not_found
Tail lights          confirmed_2_sources   not_found            not_found
Side mirrors         confirmed_2_sources   not_found            not_found
Bumpers              confirmed_2_sources   not_found            not_found
Fenders/body panels  confirmed_2_sources   not_found            not_found
Shock absorbers      confirmed_2_sources   not_found            not_found


,vehicle,part,compat_status,n_catalogue_matches,price_original,nhtsa_complaints,missing_fields
0,BYD Seagull,Brake pads,confirmed_2_sources,6,64.0,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
1,BYD Seagull,Brake discs,confirmed_2_sources,4,101.0,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
2,BYD Seagull,Cabin-air filters,not_found,0,NaN,NaN,oem_number;price_original;in_stock;wear_item;n...
3,BYD Seagull,Wiper blades,confirmed_2_sources,2,NaN,NaN,price_original;in_stock;wear_item;nhtsa_compla...
4,BYD Seagull,Headlights,confirmed_2_sources,4,418.0,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
5,BYD Seagull,Tail lights,confirmed_2_sources,6,198.0,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
6,BYD Seagull,Side mirrors,confirmed_2_sources,4,346.5,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
7,BYD Seagull,Bumpers,confirmed_2_sources,12,181.5,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
8,BYD Seagull,Fenders/body panels,confirmed_2_sources,27,1012.0,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers
9,BYD Seagull,Shock absorbers,confirmed_2_sources,8,138.0,NaN,in_stock;wear_item;nhtsa_complaints;local_sellers


## What this notebook deliberately does NOT do (do these yourself)

1. **Gezat (S02):** read the catalogue in your browser and type `gezat_listed` into `data/processed/vehicles.csv`. The `/cars` pages are disallowed for scripts.
2. **`wear_item` and `wear_basis`:** your judgement. Fill them in and give a reason.
3. **`local_sellers`:** the brief's source pack has no Ethiopian parts-seller source. Check by hand and fill `local_sellers_source` and `local_check_date`, or leave it empty and report it as unavailable.
4. **FX rate:** set `FX_RMB_TO_USD` and `FX_RATE_DATE` in the settings cell from an official source, then re-run the last cell.
5. **Review the matching:** read `parts_catalogue_raw.csv` and check that each line really belongs to the part it was matched to. The keyword rules are a starting point.
6. **Yuan Up and ID.6:** if no source lists them, the rows stay `not_found` or `fitment_unverified`. Report that as a limitation. It is a valid result.
7. **NHTSA:** complaints are component-level (for example "service brakes"), so brake pads and brake discs share one count. Open `nhtsa_components_found.csv` and adjust `NHTSA_KEYS` if the real component names differ.

## Code reuse

No code was taken from the three reference repositories in the brief. The parsers and rules here were written for this project. You must still be able to explain every important decision in this code in your presentation.